In [6]:
pip install faiss-cpu

  Using cached faiss_cpu-1.14.2-cp311-cp311-win_amd64.whl (16.1 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import faiss

In [2]:
faiss.__version__

'1.14.2'

In [3]:
bio_text = """Advance your career in data engineering and prepare for the Google Cloud Professional Data Engineer Certification exam with this program.
A Data Engineer designs, builds, and operationalizes data processing systems to solve business challenges. This certification learning path provides the advanced knowledge and practical skills required for this role, preparing you to successfully design, build, and manage data processing systems on Google Cloud.

Through a curated collection of on-demand courses, labs, and skill badges, you will gain real-world, applied experience with Google Cloud technologies.
This path focuses on the essential skills for the Data Engineer role, from designing data processing systems and building data pipelines to operationalizing machine learning models.

Upon completion, you will be equipped with the skills validated by the Professional Data Engineer certification. 

Take the next step in your professional journey and demonstrate your expertise by preparing for the Google Cloud Professional Data Engineer 
exam. #googlecloudcertified"""

In [4]:
# if we covert the above text into one single embebedding, will we be able to fulfill the requiement in terms of search ?

# suppose if we ask a question and their in one embedding that is availabe,
#  so can we say its going  to give the wholeembeddings as there is only one embedding available,
#  so it is going to give the whole text as the answer, but if we have multiple embeddings then it is going to give the relevant part of the text as the answer.


"""the whole idea is to break the text into multiple chunks and then convert those chunks into embeddings, 
so that when we ask a question, it can give us the relevant part of the text as the answer, 
rather than giving us the whole text as the answer."""

'the whole idea is to break the text into multiple chunks and then convert those chunks into embeddings, \nso that when we ask a question, it can give us the relevant part of the text as the answer, \nrather than giving us the whole text as the answer.'

In [5]:
import requests
import os
import json
import numpy as np

In [6]:
def chunk_text(text, chunk_size=50, overlap=10):
    """Break the text into chunks of specified size."""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap  # Move the start index for the next chunk
    return chunks

In [7]:
# overlaps means that we are going to have some common words between the chunks, so that we can maintain the context of the text.
# chunk_size means that we are going to break the text into chunks of 100 words, and overlap means that we are going to have 20 common words between the chunks.

In [8]:
chunks = chunk_text(bio_text)

In [9]:
chunks

['Advance your career in data engineering and prepare for the Google Cloud Professional Data Engineer Certification exam with this program. A Data Engineer designs, builds, and operationalizes data processing systems to solve business challenges. This certification learning path provides the advanced knowledge and practical skills required for this role, preparing',
 'advanced knowledge and practical skills required for this role, preparing you to successfully design, build, and manage data processing systems on Google Cloud. Through a curated collection of on-demand courses, labs, and skill badges, you will gain real-world, applied experience with Google Cloud technologies. This path focuses on the essential',
 'with Google Cloud technologies. This path focuses on the essential skills for the Data Engineer role, from designing data processing systems and building data pipelines to operationalizing machine learning models. Upon completion, you will be equipped with the skills validated

In [ ]:
url = "https://api.euron.one/api/v1/euri/embeddings"
API_KEY="sk-***********************"
MODEL_NAME="text-embedding-3-small"


In [11]:
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_KEY}"
}

In [12]:
all_embeddings = []
for i, chunk in enumerate(chunks):
    data = {
        "model": MODEL_NAME,
        "input": chunk
    }
    response = requests.post(url, headers=headers, data=json.dumps(data))
    data = response.json()
    embeddings = data['data'][0]['embedding'] 
    all_embeddings.append(embeddings)



In [13]:
len(all_embeddings)

4

In [16]:
# for i in enumerate(chunks):
#     print(i)

In [17]:
# now we have to store the data into FAISS

In [18]:
# storing the data into float32 format, because 
# FAISS works with float32 format, so we have to convert the data into float32 format before storing it into FAISS.

In [19]:
type(all_embeddings[0][0])

float

In [20]:
type(all_embeddings)

list

In [21]:
embeddings_array = np.array(all_embeddings).astype('float32')
embeddings_array

array([[ 0.03530884,  0.01802063,  0.02804565, ..., -0.01487732,
         0.00266647,  0.02035522],
       [ 0.03146362,  0.04934692,  0.04562378, ..., -0.0077858 ,
        -0.00430679, -0.01550293],
       [ 0.03372192,  0.02792358,  0.04330444, ..., -0.02285767,
        -0.00409317,  0.01474762],
       [ 0.03820801,  0.00281715,  0.03189087, ..., -0.01356506,
        -0.00402451,  0.01161194]], shape=(4, 1536), dtype=float32)

In [22]:
base_index = faiss.IndexFlatL2(embeddings_array.shape[1])



# IndexFlatL2 is a type of index in FAISS that uses the L2 distance to calculate the distance between the vectors.
# L2 means that we are going to use the L2 distance to calculate the distance between the vectors,
#  and the shape[1] means that we are going to use the number of dimensions of the embeddings as the number of dimensions of the index.

In [23]:
base_index.add(embeddings_array)

# data is now stored in FAISS, we can now perform the search on the data stored in FAISS.

In [24]:
faiss.write_index(base_index, "faiss_index.faiss")

# to store in physical file, we can use the write_index function of FAISS, which takes the index and the file name as input and stores the index in the specified file.

In [25]:
# Testing

Testing

In [26]:
query = "What is the Google Cloud Professional Data Engineer Certification exam?"

In [27]:
def embedding_text(text):
    data = {
        "model": MODEL_NAME,
        "input": text
    }
    response = requests.post(url, headers=headers, data=json.dumps(data))
    data = response.json()
    embeddings = data['data'][0]['embedding'] 
    embeddings = np.array(embeddings, dtype='float32')
    return embeddings

In [28]:
embeddings_query = embedding_text(query)
embeddings_query

array([ 0.03060913, -0.00248718,  0.03811646, ..., -0.02433777,
        0.01698303,  0.00637436], shape=(1536,), dtype=float32)

In [29]:
base_index.search(embeddings_query.reshape(1, -1), k=3)

(array([[0.3306825 , 0.3482644 , 0.46581128]], dtype=float32),
 array([[3, 0, 2]]))

In [31]:
chunks[3]

'by the Professional Data Engineer certification. Take the next step in your professional journey and demonstrate your expertise by preparing for the Google Cloud Professional Data Engineer exam. #googlecloudcertified'

In [ ]:
chunks[0]